# ⛓️ ResNet — Notes + Interview
---
> **Simple English** | **Interview Ready** | Year: 2015 | Creator: Microsoft Research (He et al.)

## 📌 What is ResNet? (Simple English)
- ResNet = **Residual Network** — solved the problem of training very deep networks
- Problem before: deeper networks were getting **worse** (degradation problem)
- ResNet solution: **skip connections** (shortcuts) that bypass layers
- The network learns **residuals** (corrections) instead of full transformations
- ResNet-50/101/152 → 50, 101, 152 layers deep!
- Won ImageNet 2015 with only 3.57% error

## 🔑 The Problem ResNet Solved
```
Deep networks WITHOUT skip connections:
→ Gradients vanish in early layers
→ Earlier layers stop learning
→ Deeper ≠ Better (before ResNet)

ResNet WITH skip connections:
→ Gradient highway: skip connection carries gradient directly back
→ Deeper = Better ✅ (up to 1000+ layers!)
```

## 🧱 Residual Block
```
Input x
  ├── Conv→BN→ReLU→Conv→BN
  └── (shortcut = identity or 1×1 conv)
  ↓
Output = F(x) + x  ← This is the key!
```

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# ── Residual Block ──
def residual_block(x, filters, stride=1):
    shortcut = x

    # Main path
    y = tf.keras.layers.Conv2D(filters,(3,3),stride,padding='same')(x)
    y = tf.keras.layers.BatchNormalization()(y)
    y = tf.keras.layers.ReLU()(y)
    y = tf.keras.layers.Conv2D(filters,(3,3),1,padding='same')(y)
    y = tf.keras.layers.BatchNormalization()(y)

    # Shortcut path — adjust dimensions if needed
    if stride != 1 or x.shape[-1] != filters:
        shortcut = tf.keras.layers.Conv2D(filters,(1,1),stride,padding='same')(x)
        shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    # ADD: F(x) + x  ← The residual connection!
    y = tf.keras.layers.Add()([y, shortcut])
    y = tf.keras.layers.ReLU()(y)
    return y

# Build ResNet-like model
inputs = tf.keras.Input(shape=(32,32,3))
x = tf.keras.layers.Conv2D(64,(3,3),padding='same')(inputs)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.ReLU()(x)

x = residual_block(x, 64)       # same size
x = residual_block(x, 128, 2)   # stride=2 → half size
x = residual_block(x, 256, 2)   # stride=2 → half size

x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(10, activation='softmax')(x)

resnet = tf.keras.Model(inputs, x, name='ResNet-Custom')
resnet.summary()
print(f"\nTotal params: {resnet.count_params():,}")

In [ ]:
# Pretrained ResNet50
resnet50 = tf.keras.applications.ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)
print("ResNet50 layers:", len(resnet50.layers))
print(f"ResNet50 params: {resnet50.count_params():,}  (~25M — much less than VGG's 138M!)")

# Add custom head
resnet50.trainable = False
x = resnet50.output
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
output = tf.keras.layers.Dense(10, activation='softmax')(x)
model_resnet = tf.keras.Model(resnet50.input, output)
print(f"Trainable params: {sum([tf.size(w).numpy() for w in model_resnet.trainable_weights]):,}")

In [ ]:
# Visualize: Normal block vs Residual block
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Normal block diagram
axes[0].text(0.5, 0.9, 'Input x', ha='center', fontsize=12, fontweight='bold')
axes[0].annotate('', xy=(0.5,0.7), xytext=(0.5,0.85), arrowprops=dict(arrowstyle='->',lw=2))
axes[0].text(0.5, 0.65, 'Conv+BN+ReLU', ha='center', fontsize=10,
             bbox=dict(boxstyle='round',facecolor='lightblue'))
axes[0].annotate('', xy=(0.5,0.45), xytext=(0.5,0.6), arrowprops=dict(arrowstyle='->',lw=2))
axes[0].text(0.5, 0.4, 'Conv+BN+ReLU', ha='center', fontsize=10,
             bbox=dict(boxstyle='round',facecolor='lightblue'))
axes[0].annotate('', xy=(0.5,0.2), xytext=(0.5,0.35), arrowprops=dict(arrowstyle='->',lw=2))
axes[0].text(0.5, 0.15, 'Output = F(x)', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Normal Block (Vanishing Gradient ❌)', fontsize=11)
axes[0].axis('off')

# Residual block diagram
axes[1].text(0.5, 0.9, 'Input x', ha='center', fontsize=12, fontweight='bold')
axes[1].annotate('', xy=(0.5,0.7), xytext=(0.5,0.85), arrowprops=dict(arrowstyle='->',lw=2))
axes[1].text(0.5, 0.65, 'Conv+BN+ReLU', ha='center', fontsize=10,
             bbox=dict(boxstyle='round',facecolor='lightgreen'))
axes[1].annotate('', xy=(0.5,0.45), xytext=(0.5,0.6), arrowprops=dict(arrowstyle='->',lw=2))
axes[1].text(0.5, 0.4, 'Conv+BN+ReLU', ha='center', fontsize=10,
             bbox=dict(boxstyle='round',facecolor='lightgreen'))
axes[1].annotate('', xy=(0.5,0.2), xytext=(0.5,0.35), arrowprops=dict(arrowstyle='->',lw=2))
axes[1].annotate('', xy=(0.7,0.2), xytext=(0.7,0.9), arrowprops=dict(arrowstyle='->',lw=2,color='red'))
axes[1].text(0.5, 0.15, 'Output = F(x) + x  ← Skip! ✅', ha='center', fontsize=11, fontweight='bold',
             color='darkgreen')
axes[1].text(0.72, 0.55, 'Skip
conn.', ha='center', fontsize=9, color='red')
axes[1].set_title('Residual Block (Skip Connection ✅)', fontsize=11)
axes[1].axis('off')
plt.tight_layout(); plt.show()

## 🗣️ Interview Q&A

**Q: What problem does ResNet solve?**
> The **degradation problem**: deeper networks were getting worse accuracy (not due to overfitting). ResNet's skip connections allow gradients to flow directly backward, enabling training of 100+ layer networks.

**Q: What is a skip/residual connection?**
> A shortcut that adds the input directly to the output of a block: `output = F(x) + x`. Network learns the residual F(x) = desired_output - x. Much easier to learn small corrections than full transformations.

**Q: Why does ResNet outperform VGG?**
> ResNet-50 (25M params) vs VGG16 (138M params): ResNet is 5× fewer parameters but better accuracy. Skip connections enable deeper training without vanishing gradients.

**Q: What is Batch Normalization in ResNet?**
> Normalizes layer inputs to have mean≈0, std≈1 during training. Placed after Conv and before activation. Makes training faster, allows higher learning rates, acts as regularizer.